<a href="https://colab.research.google.com/github/fre-caneo/Cienciadedatos/blob/Sumativa3/S4_Entrega3_CANEO_FREDDY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega 3 — Minería de datos, validación y volumen

**Asignatura:** Ciencia de Datos · Semana 4  
**Estudiante:** Freddy Caneo  
**Datos:** registros geocodificados de siniestros de tránsito de CONASET, capa seleccionada, 2021

**Pregunta de las entregas anteriores:** ¿Cómo varía la frecuencia por tramo horario y qué tipos y causas aparecen con mayor frecuencia en el tramo con más registros?

Para continuar esa pregunta, primero identifico el tramo con más siniestros. Después estudio asociaciones entre las causas y los tipos dentro de ese tramo y reviso si los resultados se mantienen durante el año.

**En Colab:** sube `mi_conjunto_explorado_2021.csv` al panel Archivos y ejecuta las celdas en orden.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ARCHIVO = Path('mi_conjunto_explorado_2021.csv')
if not ARCHIVO.exists():
    raise FileNotFoundError('Sube mi_conjunto_explorado_2021.csv a la sesión de Colab y vuelve a ejecutar.')

mio = pd.read_csv(ARCHIVO)
print(f'Archivo: {ARCHIVO.name}')
print(f'Filas: {len(mio):,} | Columnas: {mio.shape[1]}')
print(f'Período: {mio["FECHA"].min()} a {mio["FECHA"].max()}')
print(f'IDs únicos: {mio["ID"].nunique():,}')
display(mio.head(3))

Archivo: mi_conjunto_explorado_2021.csv
Filas: 57,902 | Columnas: 23
Período: 2021-01-01 a 2021-12-31
IDs únicos: 57,902


,ID,AÑO,FECHA,Dia_semana,Mes,Hora,Hora_aprox,REGION,PROVINCIA,COMUNA,...,CAUSA_NUEV,ZONA,Zonas_urb_,FALLECIDOS,TOTAL_LESI,GRAVES,MENOS_GRAV,LEVES,Tramo_horario,SEMANA_INICIO
0,1324060,2021,2021-01-01,Viernes,Enero,0:05:00,0,La Araucanía,Malleco,Purén,...,ALCOHOL EN CONDUCTOR,URBANA,Urbano,0,1,0,0,1,Madrugada (00–05),2020-12-28
1,1327715,2021,2021-01-01,Viernes,Enero,0:20:00,0,Valparaíso,Valparaíso,Valparaíso,...,OTRAS CAUSAS,URBANA,Urbano,0,0,0,0,0,Madrugada (00–05),2020-12-28
2,1323225,2021,2021-01-01,Viernes,Enero,0:25:00,0,Metropolitana de Santiago,Cordillera,Puente Alto,...,DISTRACCION DEL CONDUCTOR,URBANA,Urbano,0,0,0,0,0,Madrugada (00–05),2020-12-28


La unidad de análisis es **un siniestro por fila**. El archivo se lee con su separador original y se comprueban sus dimensiones, período e identificadores antes de analizarlo.

In [ ]:
# Revisión de calidad y campos que usaremos
mio['FECHA'] = pd.to_datetime(mio['FECHA'], errors='coerce')
mio['SEMANA_INICIO'] = pd.to_datetime(mio['SEMANA_INICIO'], errors='coerce')

faltantes = mio.isna().sum().sort_values(ascending=False)
print('Columnas con datos faltantes:')
print(faltantes[faltantes.gt(0)].to_string() if faltantes.gt(0).any() else 'Ninguna')
print(f'IDs repetidos: {mio["ID"].duplicated().sum()}')
print(f'Valores negativos en conteos de víctimas: {(mio[["FALLECIDOS", "TOTAL_LESI", "GRAVES", "MENOS_GRAV", "LEVES"]] < 0).sum().sum()}')
print(f'Registros donde TOTAL_LESI no coincide con la suma por gravedad: {(mio["TOTAL_LESI"] != mio["GRAVES"] + mio["MENOS_GRAV"] + mio["LEVES"]).sum()}')
print(f'Semanas distintas: {mio["SEMANA_INICIO"].nunique()}')

Columnas con datos faltantes:
Ninguna
IDs repetidos: 0
Valores negativos en conteos de víctimas: 0
Registros donde TOTAL_LESI no coincide con la suma por gravedad: 0
Semanas distintas: 53


En esta versión del archivo no aparecen faltantes ni IDs repetidos. La suma de lesionados graves, menos graves y leves coincide con `TOTAL_LESI`; esta comprobación será útil al revisar los registros extremos. El período ofrece semanas para hacer una validación temporal.

In [ ]:
# Distribución por horario y reglas de asociación dentro del tramo más frecuente
frecuencia = mio['Tramo_horario'].value_counts()
tramo_principal = frecuencia.idxmax()
tarde = mio[mio['Tramo_horario'].eq(tramo_principal)].copy()
print('Siniestros por tramo horario:')
display(pd.DataFrame({'siniestros': frecuencia, 'porcentaje': (100 * frecuencia / len(mio)).round(2)}))
print(f'Tramo con más registros: {tramo_principal} ({len(tarde):,})')
print('Tipos más frecuentes en ese tramo:')
display(tarde['TIPO__CONA'].value_counts().head(5).to_frame('siniestros'))
print('Causas más frecuentes en ese tramo:')
display(tarde['CAUSA_NUEV'].value_counts().head(5).to_frame('siniestros'))

# Para el conteo, cada fila aporta dos atributos: causa registrada y tipo de siniestro.
# El universo para soporte y tasas base son los siniestros del tramo principal.
N = len(tarde)
filas = []
for (causa, tipo), grupo in tarde.groupby(['CAUSA_NUEV', 'TIPO__CONA']):
    conjunto = len(grupo)
    n_causa = int(tarde['CAUSA_NUEV'].eq(causa).sum())
    tasa_tipo = tarde['TIPO__CONA'].eq(tipo).mean()
    confianza = conjunto / n_causa
    filas.append({'causa': causa, 'tipo': tipo, 'n_causa': n_causa,
                  'n_conjunto': conjunto, 'support': conjunto / N,
                  'confidence': confianza, 'lift': confianza / tasa_tipo})
reglas = pd.DataFrame(filas)
print(f'Reglas candidatas: {len(reglas)}')

Siniestros por tramo horario:


,siniestros,porcentaje
Tramo_horario,,
Tarde (12–17),21493,37.12
Noche (18–23),18028,31.14
Mañana (06–11),14412,24.89
Madrugada (00–05),3969,6.85


Tramo con más registros: Tarde (12–17) (21,493)
Tipos más frecuentes en ese tramo:


,siniestros
TIPO__CONA,
COLISION,13020
CHOQUE,6028
ATROPELLO,1508
VOLCADURA,493
OTRO TIPO,289


Causas más frecuentes en ese tramo:


,siniestros
CAUSA_NUEV,
DISTRACCION DEL CONDUCTOR,8443
IMPRUDENCIA DEL CONDUCTOR,4755
DESOBEDIENCIA A SEÑALIZACION,2763
CAUSAS NO DETERMINADAS,1634
OTRAS CAUSAS,1332


Reglas candidatas: 74


La tarde (12 a 17 horas) es el tramo con más registros. Para buscar combinaciones dentro de ese tramo, trato cada siniestro como una ficha con dos datos: la causa registrada y el tipo de siniestro. Como solo me interesan parejas de causa y tipo, puedo contar directamente cuántas veces aparece cada pareja.

El **soporte** indica qué parte de todos los siniestros de la tarde reúne ambos atributos. La **confianza** indica qué parte de los siniestros con esa causa corresponde al tipo. El **lift** compara esa última proporción con la frecuencia general del tipo en la tarde. Un lift de 1 indica que ambas proporciones son iguales. Por encima de 1, el tipo es más frecuente en los registros con esa causa. Estas medidas describen cómo aparecen juntas las categorías registradas.

In [ ]:
# Comparación de soportes y selección de reglas para interpretar
soportes = [0.01, 0.05, 0.10, 0.20]
comparacion_soporte = pd.DataFrame([
    {'soporte_minimo': s, 'casos_minimos_aprox': int(np.ceil(s * N)),
     'reglas_retenidas': int(reglas['support'].ge(s).sum())}
    for s in soportes
])
display(comparacion_soporte)
soporte_elegido = 0.05
reglas_validas = reglas[reglas['support'].ge(soporte_elegido)].copy()
print(f'Reglas con soporte de al menos {soporte_elegido:.0%}: {len(reglas_validas)}')
display(reglas_validas.sort_values('support', ascending=False).round(4))

,soporte_minimo,casos_minimos_aprox,reglas_retenidas
0,0.01,215,16
1,0.05,1075,4
2,0.10,2150,4
3,0.20,4299,1


Reglas con soporte de al menos 5%: 4


,causa,tipo,n_causa,n_conjunto,support,confidence,lift
32,DISTRACCION DEL CONDUCTOR,COLISION,8443,4939,0.2298,0.5850,0.9657
50,IMPRUDENCIA DEL CONDUCTOR,COLISION,4755,3345,0.1556,0.7035,1.1613
31,DISTRACCION DEL CONDUCTOR,CHOQUE,8443,2754,0.1281,0.3262,1.1630
26,DESOBEDIENCIA A SEÑALIZACION,COLISION,2763,2558,0.1190,0.9258,1.5283


Probé cuatro umbrales. Con 1 % aparecen 16 reglas, con 5 % y 10 % quedan cuatro, y con 20 % queda solo una. Elegí 5 % porque me interesan las combinaciones frecuentes y las cuatro que pasan ese filtro aparecen juntas en al menos 1.075 siniestros de la tarde. En este archivo, subir a 10 % no cambia la selección. Bajar a 1 % permitiría estudiar doce combinaciones adicionales que aparecen con menos frecuencia.

In [ ]:
# Tres reglas entre las causas más frecuentes y el tipo más frecuente
causas_principales = tarde['CAUSA_NUEV'].value_counts().head(3).index
tipo_principal = tarde['TIPO__CONA'].value_counts().idxmax()
seleccion = reglas_validas[reglas_validas['causa'].isin(causas_principales)
                           & reglas_validas['tipo'].eq(tipo_principal)].copy()
seleccion = seleccion.sort_values('n_causa', ascending=False)
print(f'Tipo más frecuente en la tarde: {tipo_principal}')
print(f'Frecuencia base de ese tipo en la tarde: {tarde["TIPO__CONA"].eq(tipo_principal).mean():.2%}')
display(seleccion.round(4))

Tipo más frecuente en la tarde: COLISION
Frecuencia base de ese tipo en la tarde: 60.58%


,causa,tipo,n_causa,n_conjunto,support,confidence,lift
32,DISTRACCION DEL CONDUCTOR,COLISION,8443,4939,0.2298,0.5850,0.9657
50,IMPRUDENCIA DEL CONDUCTOR,COLISION,4755,3345,0.1556,0.7035,1.1613
26,DESOBEDIENCIA A SEÑALIZACION,COLISION,2763,2558,0.1190,0.9258,1.5283


Elegí las tres causas más frecuentes para relacionarlas con el tipo más frecuente, la colisión:

- **Distracción del conductor → colisión:** 4.939 de 8.443 registros con esa causa son colisiones (58,50 %; lift 0,97). Es una combinación frecuente, pero la proporción es algo menor que la proporción general de colisiones en la tarde. Podría orientar una revisión de casos de distracción, sin asumir que la asociación es especialmente fuerte.
- **Imprudencia del conductor → colisión:** 3.345 de 4.755 son colisiones (70,35 %; lift 1,16). Es una proporción mayor que la base de la tarde; sirve para priorizar el estudio de esas circunstancias.
- **Desobediencia a señalización → colisión:** 2.558 de 2.763 son colisiones (92,58 %; lift 1,53). Es la asociación más marcada de las tres y podría justificar una revisión más detallada de lugares y señales.

La cuarta regla que supera el soporte es **distracción del conductor → choque**: 2.754 de 8.443 registros con esa causa son choques (32,62 %; lift 1,16). También puede orientar una revisión de las circunstancias de los choques relacionados con distracción.

Las categorías son las que aparecen registradas en el conjunto. Las reglas describen coincidencias; no muestran por sí solas qué provocó cada siniestro ni el riesgo por viaje.

In [ ]:
# Comparación temporal: proporción de la tarde y regla desobediencia -> colisión
semanas = np.array(sorted(mio['SEMANA_INICIO'].dropna().unique()))
corte = semanas[len(semanas) // 2 - 1]
periodos = [('Primera mitad', mio[mio['SEMANA_INICIO'] <= corte]),
            ('Segunda mitad', mio[mio['SEMANA_INICIO'] > corte])]
filas_validacion = []
for nombre, datos in periodos:
    tarde_p = datos[datos['Tramo_horario'].eq(tramo_principal)]
    causa_p = tarde_p[tarde_p['CAUSA_NUEV'].eq('DESOBEDIENCIA A SEÑALIZACION')]
    base_colision = tarde_p['TIPO__CONA'].eq('COLISION').mean()
    confianza = causa_p['TIPO__CONA'].eq('COLISION').mean()
    filas_validacion.append({'periodo': nombre, 'siniestros': len(datos),
        'tarde': len(tarde_p), 'proporcion_tarde': len(tarde_p) / len(datos),
        'casos_causa': len(causa_p), 'confidence_regla': confianza,
        'lift_regla': confianza / base_colision,
        'tipo_mas_frecuente_tarde': tarde_p['TIPO__CONA'].value_counts().idxmax(),
        'causa_mas_frecuente_tarde': tarde_p['CAUSA_NUEV'].value_counts().idxmax()})
validacion = pd.DataFrame(filas_validacion)
display(validacion.round(4))
for campo, etiqueta in [('proporcion_tarde', 'Proporción de tarde'), ('lift_regla', 'Lift de la regla')]:
    print(f'{etiqueta}: media {validacion[campo].mean():.4f} ± desviación estándar {validacion[campo].std(ddof=0):.4f}')

,periodo,siniestros,tarde,proporcion_tarde,casos_causa,confidence_regla,lift_regla,tipo_mas_frecuente_tarde,causa_mas_frecuente_tarde
0,Primera mitad,24367,9283,0.3810,1202,0.9293,1.5333,COLISION,DISTRACCION DEL CONDUCTOR
1,Segunda mitad,33535,12210,0.3641,1561,0.9231,1.5244,COLISION,DISTRACCION DEL CONDUCTOR


Proporción de tarde: media 0.3725 ± desviación estándar 0.0084
Lift de la regla: media 1.5289 ± desviación estándar 0.0045


Dividí las semanas en dos mitades. La tarde representó el 38,10 % en la primera y el 36,41 % en la segunda: media de 37,25 % y desviación estándar de 0,84 puntos porcentuales. En ambas, la colisión fue el tipo más frecuente y la distracción del conductor, la causa más frecuente.

Para desobediencia a señalización → colisión, los lifts fueron 1,5333 y 1,5244; su media fue 1,5289 y la desviación estándar, 0,0045. La asociación aparece en los dos períodos con valores parecidos. La regla se eligió mirando el conjunto completo, por lo que esta comparación sirve para revisar estabilidad temporal y no es una prueba independiente. Dos mitades de 2021 no aseguran que el patrón se mantenga en otros años.

In [ ]:
# Detección y revisión de los tres casos extremos
umbral_lesionados = mio['TOTAL_LESI'].quantile(0.99)
anomalos = mio[mio['TOTAL_LESI'] > umbral_lesionados].copy()
anomalos = anomalos.sort_values(['TOTAL_LESI', 'FALLECIDOS'], ascending=False)
top3 = anomalos.head(3).copy()
top3['suma_lesiones_por_gravedad'] = top3['GRAVES'] + top3['MENOS_GRAV'] + top3['LEVES']
top3['suma_coincide'] = top3['TOTAL_LESI'].eq(top3['suma_lesiones_por_gravedad'])

print(f'Umbral p99 de TOTAL_LESI: {umbral_lesionados:.0f} lesionados; se marca anomalía si TOTAL_LESI > p99.')
print(f'Registros sobre el umbral: {len(anomalos):,} ({len(anomalos)/len(mio):.2%})')
cols = ['ID', 'FECHA', 'Hora', 'COMUNA', 'TIPO__CONA', 'CAUSA_NUEV',
        'FALLECIDOS', 'TOTAL_LESI', 'GRAVES', 'MENOS_GRAV', 'LEVES', 'suma_coincide']
display(top3[cols])

Umbral p99 de TOTAL_LESI: 4 lesionados; se marca anomalía si TOTAL_LESI > p99.
Registros sobre el umbral: 225 (0.39%)


,ID,FECHA,Hora,COMUNA,TIPO__CONA,CAUSA_NUEV,FALLECIDOS,TOTAL_LESI,GRAVES,MENOS_GRAV,LEVES,suma_coincide
43793,1388828,2021-10-19,8:58:00,Viña del Mar,CHOQUE,DISTRACCION DEL CONDUCTOR,0,20,2,4,14,True
31080,1369204,2021-08-05,14:30:00,Talcahuano,CHOQUE,FALLAS MECANICAS,2,17,2,2,13,True
33528,1373749,2021-08-18,6:30:00,Puente Alto,COLISION,DESOBEDIENCIA A SEÑALIZACION,0,15,0,7,8,True


El percentil 99 de lesionados es 4. Marqué los casos con más de 4 lesionados: son 225 registros (0,39 %); el porcentaje no es exactamente 1 % porque varios registros comparten el valor del umbral. Elegí este corte para concentrar la revisión en los casos más extremos, aunque deja fuera otros siniestros graves.

- **ID 1388828, Viña del Mar:** 20 lesionados; el desglose suma 20. Es un siniestro muy grave y no veo un error evidente.
- **ID 1369204, Talcahuano:** 17 lesionados y 2 fallecidos; el desglose de lesiones suma 17. Podría tratarse de un hecho real de alta gravedad.
- **ID 1373749, Puente Alto:** 15 lesionados; el desglose suma 15. El registro es internamente coherente y también podría corresponder a un hecho real.

Conservaría los tres. La coherencia de las cifras no confirma por sí sola lo ocurrido: para asegurarlo habría que consultar la fuente original o antecedentes del siniestro.

In [ ]:
# Comparación de lectura completa y lectura por bloques
import json
import platform
import subprocess
import sys

# Cada ejecución usa un proceso nuevo para medir su propio pico de memoria.
programa = r'''
import json
import sys
import time
import pandas as pd

def pico_memoria_mib():
    if sys.platform == 'win32':
        import psutil
        return psutil.Process().memory_info().peak_wset / 1024**2
    import resource
    pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # macOS informa bytes. Linux (incluido Colab) informa KiB.
    return pico / (1024**2 if sys.platform == 'darwin' else 1024)

def resumir(datos):
    return (datos.groupby(['COMUNA', 'Tramo_horario'], dropna=False)
            .agg(siniestros=('ID', 'size'), fallecidos=('FALLECIDOS', 'sum'),
                 lesionados=('TOTAL_LESI', 'sum')))

ruta, estrategia = sys.argv[1:]
inicio = time.perf_counter()
if estrategia == 'completa':
    resultado = resumir(pd.read_csv(ruta))
else:
    columnas = ['ID', 'COMUNA', 'Tramo_horario', 'FALLECIDOS', 'TOTAL_LESI']
    parciales = []
    for bloque in pd.read_csv(ruta, usecols=columnas, chunksize=10_000):
        parciales.append(resumir(bloque))
    resultado = pd.concat(parciales).groupby(level=[0, 1], dropna=False).sum()
resultado = resultado.sort_index()
segundos = time.perf_counter() - inicio
pico_mib = pico_memoria_mib()

# La medición termina antes de convertir los resultados para la comparación.
filas = resultado.reset_index().to_json(orient='split', index=False)
print(json.dumps({'segundos': segundos, 'pico_mib': pico_mib, 'filas': filas}))
'''

mediciones = []
referencia = None
for repeticion in range(1, 4):
    # Alterno el orden para reducir el efecto de leer el mismo archivo seguido.
    orden = ['completa', 'bloques'] if repeticion % 2 else ['bloques', 'completa']
    for estrategia in orden:
        ejecucion = subprocess.run(
            [sys.executable, '-c', programa, str(ARCHIVO.resolve()), estrategia],
            capture_output=True, text=True, encoding='utf-8', check=True
        )
        medicion = json.loads(ejecucion.stdout)
        resultado = json.loads(medicion.pop('filas'))
        if referencia is None:
            referencia = resultado
        assert resultado == referencia, 'Los resultados agrupados difieren.'
        mediciones.append({'repeticion': repeticion, 'estrategia': estrategia, **medicion})

tabla = pd.DataFrame(mediciones)
resumen = (tabla.groupby('estrategia')[['segundos', 'pico_mib']]
           .median().rename(columns={'segundos': 'mediana_segundos',
                                    'pico_mib': 'mediana_pico_MiB'}))
print(f'Entorno: {platform.system()} | CSV: {ARCHIVO.stat().st_size / 1024**2:.2f} MiB')
display(tabla.round(3))
display(resumen.round(3))
print(f'Resultados idénticos en las seis ejecuciones ({len(referencia["data"]):,} grupos).')


Entorno: Windows | CSV: 12.80 MiB


,repeticion,estrategia,segundos,pico_mib
0,1,completa,0.086,109.398
1,1,bloques,0.075,84.984
2,2,bloques,0.073,85.016
3,2,completa,0.076,109.383
4,3,completa,0.080,109.570
5,3,bloques,0.070,85.109


,mediana_segundos,mediana_pico_MiB
estrategia,,
bloques,0.073,85.016
completa,0.080,109.398


Resultados idénticos en las seis ejecuciones (1,145 grupos).


Las dos estrategias entregaron los mismos conteos y sumas por comuna y tramo horario en las seis ejecuciones. Medí la lectura y el cálculo con el mismo reloj en ambos casos. El pico de memoria corresponde a todo el proceso e incluye Python y pandas. Como la lectura por bloques también selecciona cinco columnas, la comparación muestra el efecto de usar ambas cosas juntas.

Para un archivo que cabe con holgura en memoria, la lectura completa es sencilla. Si el archivo creciera, empezaría por leer solo las columnas necesarias y procesarlas por bloques. Si incluso los resultados parciales dejaran de caber, evaluaría una herramienta que pueda agrupar datos en disco.

En mi ejecución local, la mediana del pico bajó de 109,40 a 85,02 MiB al leer por bloques. Los tiempos quedaron cerca (0,080 y 0,073 s). La diferencia más clara en esta prueba fue la memoria. Al ejecutar en otro equipo, usaría las cifras nuevas para comparar.

In [ ]:
# Guardar el conjunto minado con una marca de revisión de casos extremos
mio['anomalia_lesionados_p99'] = mio['TOTAL_LESI'] > umbral_lesionados
salida = Path('mi_conjunto_minado_2021.csv')
mio.to_csv(salida, index=False, encoding='utf-8-sig')
print(f'Guardado: {salida.name} ({salida.stat().st_size/1024**2:.2f} MB)')
print(f'Registros marcados para revisión: {int(mio["anomalia_lesionados_p99"].sum()):,}')

Guardado: mi_conjunto_minado_2021.csv (13.13 MB)
Registros marcados para revisión: 225


Guardé todas las filas y agregué una columna que marca los casos con más lesionados que el umbral elegido. La marca indica qué registros revisar; no elimina los siniestros.

## Conclusión

La tarde (12–17 h) concentra 21.493 de los 57.902 registros de la capa seleccionada. En ese tramo, las colisiones son el tipo más frecuente y la distracción del conductor es la causa registrada más frecuente. Las reglas muestran, además, que entre los registros de la tarde con desobediencia a señalización hay una proporción especialmente alta de colisiones.

La comparación de las dos mitades permite revisar la estabilidad dentro de 2021. El hallazgo **sugiere una hipótesis para investigar**, como examinar lugares y circunstancias de las colisiones, pero no basta por sí solo para decidir una intervención: falta revisar otros años, exposición al tránsito y antecedentes de los lugares. Los conteos describen los registros disponibles y no una tasa de riesgo por viaje.

## Repositorio

[Repositorio del proyecto en GitHub](https://github.com/fre-caneo/Cienciadedatos/tree/Sumativa3).